# Dimensionality reduction

Real datasets have many columns, and many of them repeat the same information. Dimensionality
reduction compresses the features into a few informative directions, either for visualisation or as
preprocessing. This notebook covers PCA, t-SNE, and the optional UMAP on wine chemistry.

## Learning objectives

By the end of this notebook you will be able to:

- standardise features before component analysis;
- run PCA and read explained variance and loadings;
- explain the difference between a linear projection and a non-linear embedding;
- run t-SNE and (optionally) UMAP for visualisation;
- choose the number of components for a downstream model.

## Concept

**PCA** (principal component analysis) finds new axes that are linear combinations of the originals,
ordered so that the first captures the most variance. It is fast, deterministic, and invertible, and
its components are uncorrelated. The **explained variance ratio** says how much information each
component carries; the **loadings** say which original features feed each component. Because PCA
maximises variance, features must be standardised first or a large-scale column will dominate.

**t-SNE** and **UMAP** are non-linear embeddings built for visualisation. They preserve neighbourhood
structure — nearby points in the embedding were nearby in the original space — rather than global
distances. t-SNE is stochastic (set `random_state`), can be slow on large data, and its cluster
sizes and gaps are not meaningful. UMAP is faster and often preserves more global structure, but it
is an optional dependency here.

Rule of thumb: use PCA when you need a compact, reversible representation or a model input; use
t-SNE/UMAP only to look at the data, never as model features.

## Worked example

### Load and standardise

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from ds_practice import load_wine, set_seed, scatterplot

set_seed(42)
wine = load_wine()
features = [c for c in wine.columns if c != "quality"]
X = wine[features].to_numpy()
y = wine["quality"].to_numpy()
print("features:", len(features), "| samples:", len(X))

### PCA: how many components?

We fit all components and inspect the cumulative explained variance. A common choice is enough
components to reach 90%.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(random_state=42).fit(X_scaled)
cumulative = np.cumsum(pca.explained_variance_ratio_)
table = pd.DataFrame({
    "component": range(1, len(cumulative) + 1),
    "explained": pca.explained_variance_ratio_.round(3),
    "cumulative": cumulative.round(3),
})
display(table)
print("components for 90% variance:", int(np.argmax(cumulative >= 0.9) + 1))

### Loadings

Loadings tell us what each component means. The first component is dominated by features that move
together; the second often contrasts acidity against alcohol.

In [ ]:
loadings = pd.DataFrame(pca.components_[:2].T, index=features, columns=["PC1", "PC2"])
display(loadings.round(3))
print("PC1 strongest:", loadings["PC1"].abs().sort_values(ascending=False).head(4).index.tolist())
print("PC2 strongest:", loadings["PC2"].abs().sort_values(ascending=False).head(4).index.tolist())

### A 2-D PCA view

Projecting onto the first two components gives a quick look at whether quality separates.

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plot_frame = pd.DataFrame({"pc1": coords[:, 0], "pc2": coords[:, 1], "quality": y})

for q in sorted(set(y)):
    group = plot_frame[plot_frame["quality"] == q]
    scatterplot(group["pc1"], group["pc2"],
                title="Wine samples in PCA space (by quality)",
                xlabel="PC1", ylabel="PC2")
print("quality values:", sorted(set(y)))

### t-SNE

t-SNE uses a perplexity parameter (roughly, the neighbourhood size) and is slow, so we run it on the
standardised features and colour by quality. Overlapping colours confirm that quality is hard to
separate from chemistry alone.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
embedding = tsne.fit_transform(X_scaled)
tsne_frame = pd.DataFrame({"x": embedding[:, 0], "y": embedding[:, 1], "quality": y})
for q in sorted(set(y)):
    group = tsne_frame[tsne_frame["quality"] == q]
    scatterplot(group["x"], group["y"],
                title="t-SNE embedding of wine samples (by quality)",
                xlabel="t-SNE 1", ylabel="t-SNE 2")

### Optional UMAP

UMAP is not installed by default. We attempt the import and fall back to t-SNE with a clear message.

In [ ]:
try:
    import umap  # noqa: F401

    reducer = umap.UMAP(n_components=2, random_state=42)
    umap_coords = reducer.fit_transform(X_scaled)
    print("UMAP embedding shape:", umap_coords.shape)
except ImportError:
    print("umap-learn is not installed; t-SNE above is the non-linear fallback.")
    print("Install it with: pip install umap-learn")

## Exercises

1. **Component count.** Fit logistic regression on the first `n` PCA components for `n` in 1, 2, 5
   and all components, and report test accuracy. How many components preserve performance?
2. **Perplexity sensitivity.** Re-run t-SNE with `perplexity=5` and `perplexity=50` and describe how
   the apparent cluster structure changes.
3. **Standardise or not.** Run PCA without standardising and report which feature dominates the
   first component. Explain why that makes the result misleading.

## Limitations

PCA only captures linear structure and can discard directions that matter for a classification even
if they carry little variance. t-SNE distances between clusters are not meaningful, and results
change with the random seed and perplexity, so it is not a stable preprocessing step. UMAP requires
an extra dependency and its hyperparameters interact in ways that need tuning. The wine dataset is
small (about 1,600 rows), so embeddings are illustrative rather than definitive.